# Convolution

Reference:
- [CoolGPU](https://coolgpu.github.io/coolgpu_blog/github/pages/2020/10/04/convolution.html)

## Forward Pass 
Before we do the calculus, we have to precisely define the forward operation.
- Input Image ($X$): Dimensions are $C_{in} \times H \times W$.
- Kernel/Filter ($K$): Dimensions are $C_{in} \times F \times F$, and there are $C_{out}$ such filters, so the total dimensions of the kernel are $C_{out} \times C_{in} \times F \times F$.
- Output Feature Map ($Y$): Dimensions are $C_{out} \times H_{out} \times W_{out}$, where $H_{out} = \frac{H - F + 2P}{S} + 1$ and $W_{out} = \frac{W - F + 2P}{S} + 1$, where $P$ is the padding and $S$ is the stride. 

**The following explanation assumes no padding ($P=0$) and a stride of 1 ($S=1$) for simplicity.**

The standard 2D discrete convolution (technically a cross-correlation in most deep learning frameworks like PyTorch or TensorFlow) is defined as:

$$Y_{i,j} = \sum_{m=0}^{F-1} \sum_{n=0}^{F-1} X_{i+m, j+n} \cdot K_{m,n}$$

Here, $i \in [0, H_{out}-1]$ and $j \in [0, W_{out}-1]$.

During backpropagation, we are given the gradient of the scalar loss $L$ with respect to our output $Y$.
- Upstream Gradient ($\frac{\partial L}{\partial Y}$): Dimensions are identical to $Y$, which is $H_{out} \times W_{out}$.
 
Our goal is to find $\frac{\partial L}{\partial K}$ (to update our weights) and $\frac{\partial L}{\partial X}$ (to pass the gradient back to the previous layer).


## Backward Pass

### Gradient with respect to the kernel ($\frac{\partial L}{\partial K}$)

We want to know how a tiny change in a specific kernel weight $K_{m,n}$ affects the final loss $L$.

By the multivariate chain rule, the gradient is the sum of the gradients from everywhere that $K_{m,n}$ was used. 
Because $K_{m,n}$ is applied across the entire image to generate $Y$, we sum over all spatial locations in $Y$:

$$\frac{\partial L}{\partial K_{m,n}} = \sum_{i=0}^{H_{out}-1} \sum_{j=0}^{W_{out}-1} \frac{\partial L}{\partial Y_{i,j}} \cdot \frac{\partial Y_{i,j}}{\partial K_{m,n}}$$

Looking at our forward pass equation, the partial derivative of $Y_{i,j}$ with respect to a specific weight $K_{m,n}$ is just the input pixel it was multiplied by:

$$\frac{\partial Y_{i,j}}{\partial K_{m,n}} = X_{i+m, j+n}$$



Substituting this back in, we get:

$$\frac{\partial L}{\partial K_{m,n}} = \sum_{i=0}^{H_{out}-1} \sum_{j=0}^{W_{out}-1} \frac{\partial L}{\partial Y_{i,j}} \cdot X_{i+m, j+n}$$

What does this mean? Look closely at that final equation. It is exactly the formula for a convolution (cross-correlation)!  
To get the gradient of the kernel, you convolve the input image $X$ with the upstream gradient $\frac{\partial L}{\partial Y}$. 
- Dimension Check: Convolving a $H \times W$ input with an $H_{out} \times W_{out}$ gradient yields a spatial dimension of $(H - H_{out} + 1) \times (W - W_{out} + 1)$. Since $H_{out} = H - F + 1$, the output dimension perfectly matches the kernel size $F \times F$.

### Gradient with respect to the input ($\frac{\partial L}{\partial X}$)

Now, we need to know how a change in a specific input pixel $X_{i,j}$ affects the loss. 
This is required to continue backpropagation to earlier layers.

Again, using the chain rule, we sum over all locations in $Y$ that were influenced by $X_{i,j}$. 
An input pixel $X_{i,j}$ is multiplied by various kernel weights as the kernel slides over it.

$$\frac{\partial L}{\partial X_{i,j}} = \sum_{m=0}^{F-1} \sum_{n=0}^{F-1} \frac{\partial L}{\partial Y_{p,q}} \cdot \frac{\partial Y_{p,q}}{\partial X_{i,j}}$$

We need to figure out which output pixels $Y_{p,q}$ are affected. 
From the forward pass, $X$ is indexed by $(p+m, q+n)$. 
Therefore, we set $p+m = i$ and $q+n = j$, which means $p = i-m$ and $q = j-n$.

The derivative of the output with respect to the input is just the kernel weight:

$$\frac{\partial Y_{i-m, j-n}}{\partial X_{i,j}} = K_{m,n}$$

Substitute this back in:

$$\frac{\partial L}{\partial X_{i,j}} = \sum_{m=0}^{F-1} \sum_{n=0}^{F-1} \frac{\partial L}{\partial Y_{i-m, j-n}} \cdot K_{m,n}$$

(Note: If $i-m < 0$ or $j-n < 0$, or if they exceed $H_{out}-1$ and $W_{out}-1$ respectively, $\frac{\partial L}{\partial Y}$ is treated as 0. 
This naturally forms zero-padding.)

***What does this mean?***

This equation describes a "Full" Convolution. 
To get the gradient with respect to the input, you take the upstream gradient $\frac{\partial L}{\partial Y}$, pad it with zeros by $F-1$ on all sides, and convolve it with the 180-degree rotated (flipped) kernel $K$. 
The flipping occurs because of the negative signs in the indices $i-m$ and $j-n$.
- Dimension Check: A full convolution between an $H_{out} \times W_{out}$ matrix and an $F \times F$ kernel yields dimensions $(H_{out} + F - 1) \times (W_{out} + F - 1)$. Substituting $H_{out} = H - F + 1$, the output dimension simplifies perfectly back to $H \times W$.

In [1]:

import numpy as np 
np.random.seed(2026)

In [8]:
class Conv2d:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # initialize learnable parameters (weights and biases)
        self.weights = np.random.randn(out_channels, in_channels, kernel_size, kernel_size)
        self.biases = np.zeros(out_channels)

    def forward(self, input):
        """
        input shape:  (batch_size, in_channels, height, width)
        output shape: (batch_size, out_channels, output_height, output_width)
        """
        batch_size, in_channels, height, width = input.shape
        self.input_shape = input.shape

        # pad spatial dimensions
        self.padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (self.padding, self.padding), (self.padding, self.padding)),
            mode='constant', constant_values=0
        )

        # output dimensions
        output_height = (height - self.kernel_size + 2 * self.padding) // self.stride + 1
        output_width = (width - self.kernel_size + 2 * self.padding) // self.stride + 1
        self.output_shape = (batch_size, self.out_channels, output_height, output_width)
        output = np.zeros(self.output_shape)

        # cross-correlation: Y[:,oc,h,w] = sum_ic sum_m,n X[:,ic,hs+m,ws+n] * K[oc,ic,m,n] + b[oc]
        for h in range(output_height):
            for w in range(output_width):
                hs = h * self.stride
                ws = w * self.stride
                patch = self.padded_input[:, :, hs:hs+self.kernel_size, ws:ws+self.kernel_size]  # (batch_size, in_channels, kernel_size, kernel_size)
                output[:, :, h, w] += np.sum(patch[:, None, :, :, :] * self.weights[None, :, :, :, :], axis=(2, 3, 4))  # sum over in_channels and kernel dimensions
        output += self.biases[None, :, None, None]

        return output

    def backward(self, grad_output):
        """
        grad_output shape: (batch_size, out_channels, output_height, output_width)
        returns: (grad_input, grad_weights, grad_biases)
        """
        batch_size, out_channels, output_height, output_width = grad_output.shape
        _, _, input_height, input_width = self.input_shape

        padded_height = input_height + 2 * self.padding
        padded_width = input_width + 2 * self.padding
        grad_input_padded = np.zeros((batch_size, self.in_channels, padded_height, padded_width))
        grad_weights = np.zeros_like(self.weights)
        grad_biases = np.zeros_like(self.biases)

        for h in range(output_height):
            for w in range(output_width):
                hs = h * self.stride
                ws = w * self.stride
                # grad_output[:, :, h, w] has shape (batch_size, out_channels)
                g = grad_output[:, :, h, w]

                #for ic in range(self.in_channels):
                # dL/dX: scatter gradient back through the kernel
                grad_input_padded[:, :, hs:hs+self.kernel_size, ws:ws+self.kernel_size] += np.sum(
                    g[:, None, :, None, None] * self.weights.transpose(1, 0, 2, 3)[None, :, :, :, :],
                    axis=2
                )
                # dL/dK: correlate upstream gradient with saved input
                grad_weights += np.sum(
                    g[:, :, None, None, None] * self.padded_input[:, None, :, hs:hs+self.kernel_size, ws:ws+self.kernel_size],

                    axis=0
                )
                # dL/db
                grad_biases += np.sum(grad_output[:, :, h, w], axis=0)

        # remove padding from grad_input
        if self.padding > 0:
            grad_input = grad_input_padded[:, :, self.padding:-self.padding, self.padding:-self.padding]
        else:
            grad_input = grad_input_padded

        return grad_input, grad_weights, grad_biases
    

The following code is to vectorize the above naive nested loop implementation of convolution and its backward pass using `im2col` and `col2im` techniques, which are commonly used to speed up convolution operations by transforming them into matrix multiplications.

In [9]:
def forward_matmul(self, input):
    batch_size, in_channels, height, width = input.shape
    self.input_shape = input.shape

    # pad spatial dimensions
    self.padded_input = np.pad(
        input,
        ((0, 0), (0, 0), (self.padding, self.padding), (self.padding, self.padding)),
        mode='constant', constant_values=0
    )

    # output dimensions
    output_height = (height - self.kernel_size + 2 * self.padding) // self.stride + 1
    output_width = (width - self.kernel_size + 2 * self.padding) // self.stride + 1
    self.output_shape = (batch_size, self.out_channels, output_height, output_width)

    # im2col transformation: each patch is flattened into a column (batch_size, in_channels * kernel_size * kernel_size, output_height * output_width)
    # each patch is: (batch_size, in_channels, kernel_size, kernel_size) -> (batch_size, in_channels * kernel_size * kernel_size)
    # number of patches = output_height * output_width
    col = np.zeros((batch_size, in_channels * self.kernel_size * self.kernel_size, output_height * output_width))
    for h in range(output_height):
        for w in range(output_width):
            hs = h * self.stride
            ws = w * self.stride
            patch = self.padded_input[:, :, hs:hs+self.kernel_size, ws:ws+self.kernel_size]  # (batch_size, in_channels, kernel_size, kernel_size)
            col[:, :, h * output_width + w] = patch.reshape(batch_size, -1)

    self.col = col  # save im2col result for backward pass

    # reshape weights for matrix multiplication
    weights_col = self.weights.reshape(self.out_channels, -1)  # (out_channels, in_channels * kernel_size * kernel_size)
    # perform matrix multiplication and add bias
    output_col = weights_col @ col + self.biases[:, None]  # (batch_size, out_channels, output_height * output_width)
    # reshape back to output shape
    output = output_col.reshape(batch_size, self.out_channels, output_height, output_width) # (batch_size, out_channels, output_height, output_width)

    return output


def backward_matmul(self, grad_output):
    batch_size, output_channels, output_height, output_width = grad_output.shape
    _, _, input_height, input_width = self.input_shape
    
    padded_height = input_height + 2 * self.padding
    padded_width = input_width + 2 * self.padding
    
    # grad_output shape: (batch_size, out_channels, output_height, output_width)
    grad_output_col = grad_output.reshape(batch_size, self.out_channels, -1)  # (batch_size, out_channels, output_height * output_width)
    weights_col = self.weights.reshape(self.out_channels, -1)  # (out_channels, in_channels * kernel_size * kernel_size)
    
    # dL/dK: use saved im2col matrix from forward pass
    # self.col shape: (batch_size, in_channels * kernel_size * kernel_size, output_height * output_width)
    grad_weights_col = np.sum(grad_output_col @ self.col.transpose(0, 2, 1), axis=0)  # (out_channels, in_channels * kernel_size * kernel_size)
    grad_weights = grad_weights_col.reshape(self.weights.shape)

    # dL/db
    grad_biases = np.sum(grad_output_col, axis=(0, 2))

    # dL/dX: compute gradient columns, then col2im to scatter back
    grad_col = weights_col.T @ grad_output_col  # (batch_size, in_channels * kernel_size * kernel_size, output_height * output_width)
    
    # col2im: scatter gradient columns back into the padded input shape
    grad_input_padded = np.zeros((batch_size, self.in_channels, padded_height, padded_width))
    for h in range(output_height):
        for w in range(output_width):
            hs = h * self.stride
            ws = w * self.stride
            # extract the column for this spatial position and reshape to patch shape
            patch_grad = grad_col[:, :, h * output_width + w].reshape(batch_size, self.in_channels, self.kernel_size, self.kernel_size)
            # accumulate into the padded gradient (overlapping patches get summed)
            grad_input_padded[:, :, hs:hs+self.kernel_size, ws:ws+self.kernel_size] += patch_grad
    
    # remove padding from grad_input
    if self.padding > 0:
        grad_input = grad_input_padded[:, :, self.padding:-self.padding, self.padding:-self.padding]
    else:
        grad_input = grad_input_padded
    return grad_input, grad_weights, grad_biases

# patch the forward method to class Conv2d
Conv2d.forward_matmul = forward_matmul
Conv2d.backward_matmul = backward_matmul

In [10]:
# Example usage
x = np.random.randn(1, 3, 32, 32)  # batch_size=1, in_channels=3, height=32, width=32
conv_layer = Conv2d(in_channels=3, out_channels=6, kernel_size=3, stride=2, padding=1)
output = conv_layer.forward(x)
print("Output shape:", output.shape)
grad_output = np.random.randn(1, 6, 16, 16)  # same shape as output
grad_input, grad_weights, grad_biases = conv_layer.backward(grad_output)
print("Grad input shape:", grad_input.shape)
print("Grad weights shape:", grad_weights.shape)
print("Grad biases shape:", grad_biases.shape)

Output shape: (1, 6, 16, 16)
Grad input shape: (1, 3, 32, 32)
Grad weights shape: (6, 3, 3, 3)
Grad biases shape: (6,)


In [11]:
x = np.arange(1*3*32*32).reshape(1, 3, 32, 32)  # deterministic input
conv_layer = Conv2d(in_channels=3, out_channels=6, kernel_size=3, stride=2, padding=1)
output = conv_layer.forward_matmul(x)
print("Output shape:", output.shape)
print("Output:", output)



Output shape: (1, 6, 16, 16)
Output: [[[[  -651.05203569   4932.06734046   4933.42834039 ...   4948.39933964
      4949.76033957   4951.1213395 ]
   [  -734.71983533   4945.83621764   4949.87458025 ...   4994.29656895
      4998.33493156   5002.37329417]
   [  -936.29818395   5075.06382113   5079.10218374 ...   5123.52417245
      5127.56253505   5131.60089766]
   ...
   [ -3153.66001881   6496.56745957   6500.60582218 ...   6545.02781088
      6549.06617349   6553.1045361 ]
   [ -3355.23836743   6625.79506306   6629.83342567 ...   6674.25541437
      6678.29377698   6682.33213959]
   [ -3556.81671606   6755.02266656   6759.06102917 ...   6803.48301787
      6807.52138048   6811.55974309]]

  [[ -3847.13006919  -5871.18843124  -5869.70546973 ...  -5853.39289312
     -5851.90993161  -5850.42697011]
   [ -8263.33261603  -8188.75634223  -8189.25577969 ...  -8194.74959173
     -8195.24902918  -8195.74846664]
   [ -8169.01112009  -8204.73834088  -8205.23777833 ...  -8210.73159037
     -8211

In [16]:
# Example usage

x = np.random.randn(1, 3, 32, 32)  # batch_size=1, in_channels=3, height=32, width=32
conv_layer = Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=2, padding=1)

max_iters = 100
lr = 1e-02
for i in range(max_iters):
    # forward pass 
    output = conv_layer.forward(x)
    
    # loss: L = mean(|output|)
    loss = np.mean(np.abs(output))
    # grad: dL/d(output) = sign(output) / N
    grad_output = np.sign(output) / output.size
    
    # backward pass
    grad_input, grad_weights, grad_biases = conv_layer.backward(grad_output)
    
    # update weights and biases (simple SGD)
    conv_layer.weights -= lr * grad_weights
    conv_layer.biases -= lr * grad_biases
    
    # log
    if (i + 1) % 10 == 0:
        print(f"Iter {i+1}/{max_iters}, Loss: {loss:.4f}")

Iter 10/100, Loss: 4.2120
Iter 20/100, Loss: 4.2073
Iter 30/100, Loss: 4.2026
Iter 40/100, Loss: 4.1979
Iter 50/100, Loss: 4.1932
Iter 60/100, Loss: 4.1886
Iter 70/100, Loss: 4.1839
Iter 80/100, Loss: 4.1792
Iter 90/100, Loss: 4.1745
Iter 100/100, Loss: 4.1698


## Implementation Conv1D using NumPy

In [17]:
class Conv1d:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # initialize learnable parameters (weights and biases)
        self.weights = np.random.randn(out_channels, in_channels, kernel_size)
        self.biases = np.zeros(out_channels)
    
    def forward(self, x):
        """
        Args:
            x: (batch_size, in_channels, length) 
        
        """
        batch_size, in_channels, length = x.shape 
        self.input_shape = x.shape
        
        # pad spatial dimension
        if self.padding > 0:
            x = np.pad(x, ((0, 0), (0, 0), (self.padding, self.padding)), mode='constant', constant_values=0)
        self.input_padded = x  # save for backward pass 
        
        # output length
        output_length = (length - self.kernel_size + 2 * self.padding) // self.stride + 1
        output = np.zeros((batch_size, self.out_channels, output_length))
        
        # matrix multiplication: Y[:, oc, l] = sum_ic sum_k X[:, ic, ls+k] * K[oc, ic, k] + b[oc]
        for oc in range(self.out_channels):
            for l in range(output_length):
                ls = l * self.stride 
                patch = x[:, :, ls:ls+self.kernel_size]  # (batch_size, in_channels, kernel_size)
                output[:, oc, l] += np.sum(patch * self.weights[oc], axis=(1, 2))  # (batch_size,)
            output[:, oc] += self.biases[oc]
        
        return output
    
    
    def backward(self, grad_output):
        """
        Args:
            grad_output: (batch_size, out_channels, output_length)
        
        Returns:
            grad_input: (batch_size, in_channels, length)
            grad_weights: (out_channels, in_channels, kernel_size)
            grad_biases: (out_channels,)
        """
        batch_size, out_channels, output_length = grad_output.shape
        _, in_channels, length = self.input_shape
        
        # Initialize gradients
        grad_input_padded = np.zeros((batch_size, in_channels, length + 2 * self.padding))
        grad_weights = np.zeros_like(self.weights)
        grad_biases = np.zeros_like(self.biases)

        # Compute gradients
        for oc in range(self.out_channels):
            for l in range(output_length):
                ls = l * self.stride 
                g = grad_output[:, oc, l] # (batch_size,)
                
                # dL/dX: scatter gradient back through the kernel
                grad_input_padded[:, :, ls:ls+self.kernel_size] += g[:, None, None] * self.weights[oc]  # (batch_size, in_channels, kernel_size)
        
                # dL/dK: correlate upstream gradient with saved input 
                grad_weights[oc] += np.sum(g[:, None, None] * self.input_padded[:, :, ls:ls+self.kernel_size], axis=0)
            # dL/db
            grad_biases[oc] += np.sum(grad_output[:, oc, :])
            
            # remove padding from grad_input
        if self.padding > 0:
            grad_input = grad_input_padded[:, :, self.padding:-self.padding]
        else:
            grad_input = grad_input_padded
            
        return grad_input, grad_weights, grad_biases

In [18]:
# run a simple example for Conv1d
x = np.arange(1, 11).reshape(1, 1, 10)  # (batch_size=1, in_channels=1, length=10)
conv = Conv1d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=0) 

conv.weights = np.array([[[0.1, 0.1, 0.1]]])  # (out_channels=1, in_channels=1, kernel_size=3)
conv.biases = np.array([0.0])  # (out_channels=1)

output = conv.forward(x)
print("Input:", x)
print("Output:", output)

Input: [[[ 1  2  3  4  5  6  7  8  9 10]]]
Output: [[[0.6 0.9 1.2 1.5 1.8 2.1 2.4 2.7]]]


In [19]:
# an example: learn filter to null the input signal
x = np.random.randn(1, 1, 10)  # (batch_size=1, in_channels=1, length=10)
conv = Conv1d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=0)

max_iters = 100
lr = 1e-02
for i in range(max_iters):
    # forward pass 
    output = conv.forward(x)
    
    # loss: L = mean(|output|)
    loss = np.mean(np.abs(output))
    # grad: dL/d(output) = sign(output) / N
    grad_output = np.sign(output) / output.size
    
    # backward pass
    grad_input, grad_weights, grad_biases = conv.backward(grad_output)
    
    # update weights and biases (simple SGD)
    conv.weights -= lr * grad_weights
    conv.biases -= lr * grad_biases
    
    # log
    if (i + 1) % 10 == 0:
        print(f"Iter {i+1}/{max_iters}, Loss: {loss:.4f}")

Iter 10/100, Loss: 0.4445
Iter 20/100, Loss: 0.3534
Iter 30/100, Loss: 0.2675
Iter 40/100, Loss: 0.1866
Iter 50/100, Loss: 0.1084
Iter 60/100, Loss: 0.0298
Iter 70/100, Loss: 0.0172
Iter 80/100, Loss: 0.0111
Iter 90/100, Loss: 0.0136
Iter 100/100, Loss: 0.0136
